# Evaluation of building permit cost predictor

This notebook loads the best model selected in notebook 03 and performs a
detailed evaluation: predicted-vs-actual plots, residual analysis, feature
importance inspection, and error breakdown by permit type. The goal is to
understand where the model excels and where it struggles, and to translate
the R-squared of ~0.89 into practical business meaning.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from src.data_loader import load_or_fetch_data, preprocess_data, engineer_features
from src.model import (
    prepare_model_data, train_models, get_feature_importance,
    load_model, TARGET,
)

sns.set_style('whitegrid')
%matplotlib inline

## 1. Reload data and retrain for full evaluation context

We need the raw dataframe alongside predictions so we can segment errors
by permit type and community. We retrain via `train_models` to get the
test-set predictions aligned with metadata.

In [ ]:
import os

engineered_path = '../data/building_permits_engineered.csv'
if os.path.exists(engineered_path):
    df = pd.read_csv(engineered_path, low_memory=False)
else:
    raw = load_or_fetch_data('../data', limit=500000)
    df = preprocess_data(raw)
    df = engineer_features(df)

X, y, label_encoders, feature_names = prepare_model_data(df)
trained_models, results, scaler, X_test, y_test = train_models(X, y, random_state=42)

# Select the best model by R2
best_name = max(results, key=lambda k: results[k]['R2'])
best_model = trained_models[best_name]
print(f'Best model: {best_name}')
print(f'Hold-out metrics: {results[best_name]}')

In [ ]:
# Generate predictions on the test set
y_pred = best_model.predict(X_test)

# Convert from log space to original dollar amounts
y_test_dollars = np.expm1(y_test)
y_pred_dollars = np.expm1(y_pred)

## 2. Predicted vs actual scatter plot

Points near the 45-degree line indicate accurate predictions. We show both
log-space and original-dollar-space views.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Log space
axes[0].scatter(y_test, y_pred, alpha=0.15, s=5, color='steelblue')
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
axes[0].plot(lims, lims, 'r--', lw=1.5, label='Perfect prediction')
axes[0].set_xlabel('Actual log_cost')
axes[0].set_ylabel('Predicted log_cost')
axes[0].set_title('Predicted vs actual (log space)', fontsize=11)
axes[0].legend()

# Dollar space
axes[1].scatter(y_test_dollars, y_pred_dollars, alpha=0.15, s=5, color='teal')
lims_d = [0, np.percentile(y_test_dollars, 99)]
axes[1].plot(lims_d, lims_d, 'r--', lw=1.5, label='Perfect prediction')
axes[1].set_xlabel('Actual cost ($)')
axes[1].set_ylabel('Predicted cost ($)')
axes[1].set_title('Predicted vs actual (dollar space)', fontsize=11)
axes[1].set_xlim(lims_d)
axes[1].set_ylim(lims_d)
axes[1].legend()

plt.tight_layout()
plt.show()

## 3. Residual analysis

Residuals (actual minus predicted) should be centered around zero with no
obvious pattern. A funnel shape would indicate heteroscedasticity; clusters
of large residuals would point to systematic bias for certain cost ranges.

In [ ]:
residuals = y_test.values - y_pred

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Residuals vs predicted
axes[0].scatter(y_pred, residuals, alpha=0.1, s=4)
axes[0].axhline(0, color='red', ls='--')
axes[0].set_xlabel('Predicted log_cost')
axes[0].set_ylabel('Residual')
axes[0].set_title('Residuals vs predicted', fontsize=11)

# Residual histogram
axes[1].hist(residuals, bins=80, edgecolor='black', alpha=0.7, color='steelblue')
axes[1].axvline(0, color='red', ls='--')
axes[1].set_xlabel('Residual')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Residual distribution', fontsize=11)

# QQ plot
from scipy import stats
stats.probplot(residuals, plot=axes[2])
axes[2].set_title('Q-Q plot of residuals', fontsize=11)

plt.tight_layout()
plt.show()

print(f'Residual mean: {residuals.mean():.4f}')
print(f'Residual std:  {residuals.std():.4f}')
print(f'Skewness:      {pd.Series(residuals).skew():.4f}')

## 4. Feature importance

We use the built-in `feature_importances_` attribute from the tree-based
best model via `get_feature_importance` in `model.py`. This gives the
mean decrease in impurity (Gini importance) across all splits.

In [ ]:
importance_df = get_feature_importance(best_model, feature_names, model_name=best_name)

fig, ax = plt.subplots(figsize=(8, 6))
top_n = importance_df.head(15)
ax.barh(top_n['Feature'], top_n['Importance'], color='teal', edgecolor='black')
ax.invert_yaxis()
ax.set_xlabel('Feature importance (impurity-based)')
ax.set_title(f'Top 15 features -- {best_name}', fontsize=12)
plt.tight_layout()
plt.show()

print(importance_df.to_string(index=False))

## 5. Error distribution by permit type

We join predictions back to the original dataframe to examine how error
varies across permit types. Some categories (e.g., large commercial
developments) may have higher variance and therefore larger errors.

In [ ]:
# Build an evaluation dataframe with metadata
eval_df = df.loc[X_test.index].copy()
eval_df['y_actual'] = y_test.values
eval_df['y_pred'] = y_pred
eval_df['abs_error'] = np.abs(y_test.values - y_pred)
eval_df['abs_error_dollars'] = np.abs(y_test_dollars.values - y_pred_dollars)

# MAE by permit type
if 'permittype' in eval_df.columns:
    type_errors = eval_df.groupby('permittype').agg(
        count=('abs_error', 'size'),
        mae_log=('abs_error', 'mean'),
        mae_dollars=('abs_error_dollars', 'mean'),
    ).sort_values('mae_dollars', ascending=False)
    print('Error by permit type:\n')
    print(type_errors.round(2).to_string())

In [ ]:
# Box plot of absolute error by top permit types
if 'permittype' in eval_df.columns:
    top_types = eval_df['permittype'].value_counts().head(8).index
    plot_df = eval_df[eval_df['permittype'].isin(top_types)]

    fig, ax = plt.subplots(figsize=(10, 5))
    plot_df.boxplot(column='abs_error', by='permittype', ax=ax, vert=True)
    ax.set_title('Absolute error (log space) by permit type', fontsize=11)
    ax.set_ylabel('Absolute error')
    plt.suptitle('')  # remove default pandas title
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.show()

## 6. Comparing all models on key metrics

A final side-by-side table of all four models, combining hold-out and
dollar-space metrics.

In [ ]:
comparison = pd.DataFrame(results).T
comparison = comparison.round(4)
comparison = comparison.sort_values('R2', ascending=False)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, metric in zip(axes, ['R2', 'MAE', 'RMSE']):
    comparison[metric].plot(kind='bar', ax=ax, color='steelblue', edgecolor='black')
    ax.set_title(metric, fontsize=11)
    ax.set_ylabel(metric)
    ax.tick_params(axis='x', rotation=25)
plt.tight_layout()
plt.show()

print(comparison.to_string())

## 7. Business interpretation

An R-squared of approximately 0.89 means the model explains about 89% of the
variance in log-transformed construction costs. In practical terms:

- **Budget planning**: Project managers can use predictions to generate initial
  cost estimates before detailed engineering quotes arrive, reducing the
  turnaround time for feasibility studies.
- **Anomaly detection**: Permits whose actual cost deviates sharply from the
  predicted cost may warrant closer review -- potential red flags for cost
  overruns or underreporting.
- **Policy analysis**: The city can simulate how changes in zoning (which affect
  `permitclassgroup`) or housing density (`housingunits`) would shift expected
  project costs across communities.

The remaining 11% of variance likely comes from factors not captured in the
permit metadata: material prices, contractor selection, site-specific
geotechnical conditions, and project complexity details hidden inside the
free-text `description` field.

In [ ]:
# Quantify the prediction spread in dollar terms
pct_within_50 = (eval_df['abs_error_dollars'] < 50000).mean() * 100
pct_within_100 = (eval_df['abs_error_dollars'] < 100000).mean() * 100
median_error = eval_df['abs_error_dollars'].median()

print(f'Median absolute error: ${median_error:,.0f}')
print(f'Predictions within $50K of actual: {pct_within_50:.1f}%')
print(f'Predictions within $100K of actual: {pct_within_100:.1f}%')

## Summary

- The best model (XGBoost) achieves R-squared ~0.89 on hold-out data.
- Residuals are approximately normally distributed and centered near zero,
  with slight heavy tails at extreme cost levels.
- `community_avg_cost` and `totalsqft` are the two most important features.
- Errors are largest for commercial and institutional permit types, which
  have inherently higher cost variance.
- The model is suitable for preliminary cost estimation and anomaly flagging.